# Goal consistency: cached target versus fresh goal

Select a **GPU runtime**, then **Run all**. This is a focused diagnostic: it loads the frozen image encoder and checks a handful of images. It executes **zero simulation actions** and **zero training updates**. The existing curriculum and source caches are read without modification; results go to a new timestamped folder under `goal_consistency_audit_v1`.

We are investigating why the raw cost at the physical goal `(50,42)` is not close to zero. A nonzero cost identifies differing tokens; it does not, by itself, identify the cause or establish that earlier planning outcomes are invalid.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil, subprocess, sys, time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Choose a GPU runtime for the fresh encoder checks.'
DRIVE = Path('/content/drive/MyDrive')
CURRICULUM = DRIVE / 'datasets/curriculum_cost_adapter/multiband_curriculum_v1'
CLEAN = DRIVE / 'datasets/curriculum_cost_adapter/clean_pair_v1'
ENCODER_DRIVE = DRIVE / 'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
OUTPUT = DRIVE / 'datasets/curriculum_cost_adapter/goal_consistency_audit_v1' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
REPO = Path('/content/i-jepa-wm-goal-audit')
REPO_URL = 'https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT = '7367ee2b92be83e85e49ca696c4051cdce6ecec5'
DINO_ROOT = Path('/content/dino_wm')
DINO_COMMIT = '0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER = Path('/content/curriculum-goal-audit-artifacts/encoder.pth.tar')

def sha256(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            result.update(block)
    return result.hexdigest()

def shell(command):
    command = [str(arg) for arg in command]
    print('$', ' '.join(command), flush=True)
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            code = process.wait()
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
    if code: raise subprocess.CalledProcessError(code, command)

assert CURRICULUM.is_dir() and CLEAN.is_dir(), 'The completed curriculum and clean-pair source must be in Drive.'
assert ENCODER_DRIVE.is_file(), f'Missing encoder checkpoint: {ENCODER_DRIVE}'
print('GPU:', torch.cuda.get_device_name())
print('Existing experiment:', CURRICULUM)
print('New diagnostic output:', OUTPUT)


## Load the same encoder and pinned renderer

This uses the encoder checkpoint already in Drive. The world model is not loaded. Setup may take a few minutes to copy the checkpoint and install dependencies; the actual check performs only a few single-image and eight-image encoder passes.


In [ ]:
if not REPO.exists():
    shell(['git', 'clone', '--branch', 'curriculum-cost-adapter', '--single-branch', REPO_URL, REPO])
else:
    assert subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip() == REPO_URL
    assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip(), 'Audit clone has local changes.'
    shell(['git', '-C', REPO, 'fetch', 'origin', 'curriculum-cost-adapter'])
assert SOURCE_COMMIT, 'Use the published notebook with a pinned diagnostic revision.'
shell(['git', '-C', REPO, 'checkout', '--detach', SOURCE_COMMIT])
shell([sys.executable, '-m', 'pip', 'install', '-q', 'gym==0.26.2', 'omegaconf', 'pyyaml', 'scipy', 'einops', 'matplotlib', 'pillow'])
if not DINO_ROOT.exists():
    shell(['git', 'clone', 'https://github.com/gaoyuezhou/dino_wm.git', DINO_ROOT])
assert not subprocess.check_output(['git', '-C', str(DINO_ROOT), 'status', '--porcelain', '--untracked-files=no'], text=True).strip(), 'DINO source has tracked changes.'
shell(['git', '-C', DINO_ROOT, 'checkout', DINO_COMMIT])
expected_encoder = json.loads((CURRICULUM / 'experiment.json').read_text())['sources']['models']['encoder']
ENCODER.parent.mkdir(parents=True, exist_ok=True)
if not ENCODER.exists() or sha256(ENCODER) != expected_encoder:
    shutil.copy2(ENCODER_DRIVE, ENCODER)
assert sha256(ENCODER) == expected_encoder, 'Encoder differs from the experiment checkpoint.'
print('Ready: only the frozen image encoder will be loaded; no world-model checkpoint is needed.')


## Run the controlled comparisons

1. Check the source goal, its copied target, the grid's embedded target, and the exact saved grid point `(50,42)`.
2. Compare pixels from repeated goal renders, the grid-rendering path, a real observation reset to the goal, and another reset seed.
3. Encode identical goal pixels singly and in a batch of eight; also encode the actual eight-position grid batch containing the goal. Compare both with the saved tokens.
4. Repeat with strict FP32 arithmetic (TF32 disabled), recording the settings. "Current FP32" means this new runtime's current settings, not a reconstruction of every historical setting. The original run did not record all arithmetic/backend flags.

The original cached goal image was not saved, only its tokens. Fresh pixel agreement therefore cannot prove what its historical pixels were.


In [ ]:
started = time.monotonic()
command = [sys.executable, '-m', 'src.mylonas_ijepa_wm.curriculum_goal_audit',
    '--curriculum-dir', CURRICULUM, '--clean-dir', CLEAN, '--output-dir', OUTPUT,
    '--encoder-checkpoint', ENCODER, '--dino-wm-root', DINO_ROOT, '--device', 'cuda']
# Run from the pinned clone. The audit reads existing files and writes only to OUTPUT.
command = [str(arg) for arg in command]
with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        result = process.wait()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=10)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
if result: raise subprocess.CalledProcessError(result, command)
print(f'Fresh-encoding diagnostic took {(time.monotonic()-started)/60:.1f} minutes.')


## Read the results

- **goal_self:** must be zero; this is an arithmetic sanity check.
- **goal_copy / grid_target_copy:** test whether all saved target copies agree.
- **grid_goal_vs_cached_goal:** reproduces the nonzero value in the existing map.
- **fresh_single_vs_fresh_repeat:** establishes repeated-encoding variability for identical pixels.
- **single_vs_repeated_image_batch8:** isolates a batch-size effect with identical pixels, using the same batch slot as the goal in the grid.
- **single_vs_grid_batch8:** tests the grid batch used in the experiment; read alongside the pixel checks.
- **identical_vs_mixed_batch_goal_slot:** compares that same slot when its neighbors are identical goal images versus the actual grid images.
- **within_identical_image_batch8:** checks all eight outputs against the first slot, including the goal slot.
- **fresh_single_vs_cached_goal / fresh_grid_batch_vs_saved_grid_goal:** show which saved representations the fresh paths reproduce.

Compare differences to the repeated-single variability and to the original saved mismatch. Strict FP32 reducing a batch difference would implicate arithmetic settings in this runtime; it would not by itself prove which settings the historical run used. Agreement with saved tokens helps establish that link.

No target will be automatically replaced. Return this executed notebook so we can identify the cause before changing the experiment.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

report = json.loads((OUTPUT / 'report.json').read_text())
saved = report['saved']
print('SAVED ARTIFACTS')
for name in ('goal_self', 'goal_copy', 'grid_target_copy', 'grid_goal_vs_cached_goal'):
    row = saved[name]
    print(f"{name:<30} MSE={row['mse']:.9g}; exact={row['exact']}")
print('Stored raw cost at (50,42):', saved['stored_raw_goal_cost'])
print('Original experiment artifacts unchanged:', report['source_hashes_unchanged'])
print('Primitive actions executed:', report['primitive_interactions'])

print('\nPIXEL CHECKS')
for mode, data in report['probes'].items():
    print(mode, {name: row['exact'] for name, row in data['pixel_comparisons'].items()})

print('\nENCODING CHECKS (MSE)')
print(f"{'Comparison':<48} {'current fp32':>16} {'strict fp32':>16}")
for name, current in report['probes']['current_fp32']['token_comparisons'].items():
    strict = report['probes']['strict_fp32']['token_comparisons'][name]
    print(f"{name:<48} {current['mse']:>16.9g} {strict['mse']:>16.9g}")
print('\nArithmetic settings:', {mode: data['arithmetic'] for mode, data in report['probes'].items()})

payload = torch.load(OUTPUT / 'probe_tensors.pt', map_location='cpu', weights_only=True)
frames = payload['current_fp32']['frames']
fig, axes = plt.subplots(1, 4, figsize=(12, 3.5), constrained_layout=True)
for ax, name, title in zip(axes[:3], ('render_goal', 'grid_batch_goal', 'observation_at_goal'),
                         ('Fresh goal image', 'Grid image at (50,42)', 'Actual observation at goal')):
    ax.imshow(frames[name].numpy()); ax.set_title(title, fontsize=10); ax.axis('off')
delta = (frames['render_goal'].float() - frames['grid_batch_goal'].float()).abs().amax(dim=-1)
image = axes[3].imshow(delta.numpy(), cmap='magma', vmin=0, vmax=max(1, float(delta.max())))
axes[3].set_title(f'Absolute pixel difference\nmaximum = {float(delta.max()):g}', fontsize=10)
axes[3].axis('off')
fig.colorbar(image, ax=axes[3], shrink=0.7, label='Pixel levels')
fig.savefig(OUTPUT / 'fresh_goal_pixels.png', dpi=150)
plt.show(); plt.close(fig)
print('Full numeric report:', OUTPUT / 'report.json')
print('Fresh image/tokens bundle:', OUTPUT / 'probe_tensors.pt')
